# Gold: Urbanization Analytics

Default lakehouse: `gold_lh`. Attaches `silver_lh` read-only. **Never attaches `bronze_lh`** —
physically enforces the Bronze->Silver->Gold isolation mandate in root `CLAUDE.md`.
Produces per-entity metrics plus continent/WHO-region rollups.

In [1]:
try:
    import notebookutils  # noqa: F401  (only importable inside Fabric)
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if not IN_FABRIC:
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession
    builder = (
        SparkSession.builder.appName("urbanisation_prospects_gold")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

DEFAULT_LAKEHOUSE = "gold_lh"
SILVER_LAKEHOUSE = "silver_lh"
# Paths are relative to this notebook's own directory (Jupyter's default cwd), i.e. repo_root/notebooks/
LOCAL_LAKEHOUSE_ROOT = "../lakehouse"


def table_path(lakehouse: str, table: str) -> str:
    if IN_FABRIC:
        mount = "default" if lakehouse == DEFAULT_LAKEHOUSE else lakehouse
        return f"/lakehouse/{mount}/Tables/{table}"
    return f"{LOCAL_LAKEHOUSE_ROOT}/{lakehouse}/Tables/{table}"


def read_delta(lakehouse: str, table: str):
    return spark.read.format("delta").load(table_path(lakehouse, table))


def write_delta(lakehouse: str, table: str, df, mode: str = "overwrite"):
    (
        df.write.format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .save(table_path(lakehouse, table))
    )

In [2]:
from pyspark.sql import Window
from pyspark.sql import functions as F

silver_df = read_delta(SILVER_LAKEHOUSE, "population_urbanization")

share_safe = F.col("total_population").isNotNull() & (F.col("total_population") != 0)

entity_year = (
    silver_df.withColumn(
        "urban_share_pct",
        F.when(share_safe, F.col("urban_population") / F.col("total_population") * 100).otherwise(
            F.lit(None).cast("double")
        ),
    ).withColumn(
        "rural_share_pct",
        F.when(share_safe, F.col("rural_population") / F.col("total_population") * 100).otherwise(
            F.lit(None).cast("double")
        ),
    )
)

entity_window = Window.partitionBy("entity").orderBy("year")
prev_urban = F.lag("urban_population").over(entity_window)

entity_year = entity_year.withColumn(
    "urban_population_yoy_growth_pct",
    F.when(
        prev_urban.isNotNull() & (prev_urban != 0),
        (F.col("urban_population") - prev_urban) / prev_urban * 100,
    ).otherwise(F.lit(None).cast("double")),
)

In [3]:
urbanization_by_entity_year = entity_year.select(
    "entity",
    "year",
    "urban_population",
    "rural_population",
    "total_population",
    "urban_share_pct",
    "rural_share_pct",
    "urban_population_yoy_growth_pct",
)

write_delta(DEFAULT_LAKEHOUSE, "urbanization_by_entity_year", urbanization_by_entity_year, mode="overwrite")
print(f"Gold urbanization_by_entity_year rows: {urbanization_by_entity_year.count()}")

Gold urbanization_by_entity_year rows: 27573


In [4]:
def rollup(df, group_col):
    return df.groupBy(group_col, "year").agg(
        F.sum("urban_population").alias("total_urban_population"),
        F.sum("rural_population").alias("total_rural_population"),
        F.avg("urban_share_pct").alias("avg_urban_share_pct"),
    )


continent_rollup = rollup(entity_year.filter(F.col("continent").isNotNull()), "continent")
write_delta(DEFAULT_LAKEHOUSE, "urbanization_by_continent_year", continent_rollup, mode="overwrite")
print(f"Gold urbanization_by_continent_year rows: {continent_rollup.count()}")

who_region_rollup = rollup(entity_year.filter(F.col("who_region").isNotNull()), "who_region")
write_delta(DEFAULT_LAKEHOUSE, "urbanization_by_who_region_year", who_region_rollup, mode="overwrite")
print(f"Gold urbanization_by_who_region_year rows: {who_region_rollup.count()}")

Gold urbanization_by_continent_year rows: 606


Gold urbanization_by_who_region_year rows: 606
